### 玩家关卡失败后流失率

In [1]:
import datetime
import sys
# path为tools的上层文件夹
sys.path.append('D:\milimili')
import numpy as np
import pandas as pd
from configs import TE_app_token
from tools.数数查询 import get_sqldata
token=TE_app_token['Doomsday Vanguard']

In [2]:
# 流失玩家定义：
# 最后一次登录时间距今超过7天
# 24年新用户

In [3]:
sql1="""
select "#distinct_id","last_login_time"
from ta.v_user_33 
where date_diff('day',"last_login_time", CURRENT_TIMESTAMP) >=7
and "first_login_time" is not NULL 
and to_char("first_login_time",'yyyy-mm-dd') > '2024-01-01'
"""

In [4]:
liushi_user = get_sqldata(token,sql1)

In [5]:
liushi_user.columns = ['id','最后一次登录时间']

In [6]:
# 玩家最后一次游戏失败对应关卡、失败次数

In [19]:
sql2="""
select  "level_name" ,"#distinct_id",count("#event_time") as "关卡失败次数", max("#event_time") as "最后一次闯关时间"
from ta.v_event_33 
where "$part_event"= 'Game_Level_Finish'
and "if_clearance" = False
and "#country_code"= 'JP'
and "$part_date" between '2024-01-01' and '2024-03-11'
group by  "level_name","#distinct_id"
"""

In [21]:
user_last_level = get_sqldata(token,sql2)

In [22]:
user_last_level.columns = ['最后一次关卡','id1','失败次数','col1']

In [23]:
user_last_level

,最后一次关卡,id1,失败次数,col1
0,ChapterName_021,66edea65c4f043d2,1,2024-03-08 21:32:28.730
1,ChapterName_046,AC1D4519-36E7-4180-8E52-44E3FE5DC9B7,39,2024-03-11 18:00:00.297
2,ChapterName_009,FA084813-ADB6-4E77-AD84-F3A9EE7107AC,3,2024-03-09 00:39:10.238
3,ChapterName_013,85dda4202ecc3f34,3,2024-03-08 03:09:49.090
4,ChapterName_011,b887964d2102275e,1,2024-03-08 01:47:06.069
...,...,...,...,...
525670,ChapterName_009,3ca99090e80cbd3d,1,2024-02-15 21:24:12.149
525671,ChapterName_005,fa234a98a4adc51e,1,2024-02-15 20:44:02.588
525672,ChapterName_003,362E7C04-89CE-49C8-9EE7-624BBBBF73C6,1,2024-02-15 21:45:53.578
525673,ChapterName_011,74a51f2b6e78befb,1,2024-02-15 21:41:07.164


In [11]:
liushi_user

,id,最后一次登录时间
0,null,2024-03-16 05:35:19.000
1,65035d5761a333b6,2024-02-07 03:44:15.000
2,f61e223ba74a8905,2024-03-18 11:38:16.000
3,d5892740aaee185c,2024-01-18 10:45:06.000
4,77ac5e1646d556d5,2024-01-05 07:36:02.000
...,...,...
278172,4fa3a0e617da5acf,2024-03-14 13:22:43.000
278173,0436ead53dcecb7f,2024-03-14 13:25:20.000
278174,null,2024-03-14 13:26:40.000
278175,021ded80cdf81f86,2024-03-14 13:33:44.000


In [24]:
res1 = user_last_level[['id1','最后一次关卡','失败次数']].merge(liushi_user,how='left',left_on='id1',right_on = 'id')

In [25]:
res1=res1.groupby(by=['最后一次关卡','失败次数'])[['id1','id']].agg({
    'id1':'nunique',
    'id':'nunique'
}).reset_index()

In [26]:
res1['流失率']=res1['id']/res1['id1']

In [106]:
res1.to_excel('关卡失败流失情况.xlsx',index=False)

In [27]:
res1

,最后一次关卡,失败次数,id1,id,流失率
0,ChapterName_001,1,10493,9062,0.863623
1,ChapterName_001,10,5,5,1.000000
2,ChapterName_001,11,3,1,0.333333
3,ChapterName_001,1188,1,0,0.000000
4,ChapterName_001,12,9,4,0.444444
...,...,...,...,...,...
3129,ChapterName_130,2,1,0,0.000000
3130,ChapterName_130,23,1,0,0.000000
3131,ChapterName_130,3,2,0,0.000000
3132,ChapterName_130,5,2,1,0.500000


### 玩家关卡失败后氪金情况

In [5]:
# 玩家付费

In [34]:
sql4="""
select "#distinct_id","#event_time","max_chapter_id"
from ta.v_event_33 
where "$part_event" = 'In_appPurchases_BuySuccess' 
and "$part_date" between '2024-01-01' and '2024-03-11'
"""

In [35]:
ng_user = get_sqldata(token,sql4)

In [63]:
ng_user.columns=['id1','内购时间','关卡']

In [64]:
ng_user=ng_user[ng_user['关卡']!='null']

In [65]:
ng_user['关卡']=ng_user['关卡'].astype('int')

In [7]:
# 玩家关卡失败情况

In [8]:
sql5="""
select  "level_name" ,"#distinct_id",count("#event_time") as "关卡失败次数", max("#event_time") as "最后一次闯关时间"
from ta.v_event_33 
where "$part_event"= 'Game_Level_Finish'
and "if_clearance" = False
and "#country_code"= 'JP'
and "$part_date" between '2024-01-01' and '2024-03-11'
group by  "level_name","#distinct_id"
order by "#distinct_id", "level_name"
"""

In [13]:
user_level = get_sqldata(token,sql5)

In [19]:
user_level.columns = ['关卡','id','关卡失败次数','最后闯关时间']

In [48]:
user_level=user_level.iloc[0:525674,:]

In [50]:
user_level['关卡']= user_level['关卡'].str.replace('ChapterName_','').astype('int')

C:\Users\admin\AppData\Local\Temp\ipykernel_20792\3610303703.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_level['关卡']= user_level['关卡'].str.replace('ChapterName_','').astype('int')


In [52]:
user_level

,关卡,id,关卡失败次数,最后闯关时间
0,3,00001d334c46cdc2,1,2024-02-08 15:32:57.666
1,20,00001d334c46cdc2,2,2024-02-15 02:14:12.166
2,3,000087e9c79bcd3d,2,2024-02-13 22:41:03.002
3,3,0001E370-A5F8-48B1-93C0-AD7BB96BE088,2,2024-02-27 14:06:14.568
4,6,0001E370-A5F8-48B1-93C0-AD7BB96BE088,2,2024-02-27 19:03:10.826
...,...,...,...,...
525669,8,fff9056503927b46,1,2024-01-03 20:47:15.036
525670,2,fffafcadc392037b,1,2024-02-12 23:39:39.976
525671,12,ffffc208a898d9cb,1,2024-01-07 03:22:45.742
525672,13,ffffc208a898d9cb,3,2024-01-08 19:02:43.305


In [67]:
res2=user_level.merge(ng_user,how='left',left_on=['id','关卡'],right_on=['id1','关卡'])

In [70]:
res2=res2.groupby(by=['关卡','关卡失败次数'])[['id','id1']].agg({
    'id':'nunique',
    'id1':'nunique'
}).reset_index()

In [71]:
res2['失败内购率']=res2['id1']/res2['id']

In [107]:
res2.to_excel('关卡失败付费情况.xlsx',index=False)

### 大R成长路径（关卡）

In [21]:
sql6="""
select "#distinct_id","max_chapter_id","pay_amount"
from ta.v_event_33 
where  "$part_event" = 'In_appPurchases_BuySuccess'
and "$part_date" between '2024-01-01' and '2024-03-19'
and "#country_code" = 'JP'
and "#distinct_id" in (
select "#distinct_id" from ta.v_user_33 where "total_pay_amount" >=300
)
"""

In [22]:
ng300_info = get_sqldata(token,sql6)

In [76]:
ng300_info=ng300_info.iloc[0:42044,:]

In [81]:
ng300_info['金额']=ng300_info['金额'].astype('float')

C:\Users\admin\AppData\Local\Temp\ipykernel_20792\1300102423.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ng300_info['金额']=ng300_info['金额'].astype('float')


In [74]:
ng300_info.columns=['id','关卡','金额']

In [82]:
ng300_info['金额USD']=ng300_info['金额']/150.30299

C:\Users\admin\AppData\Local\Temp\ipykernel_20792\1637585750.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ng300_info['金额USD']=ng300_info['金额']/150.30299


In [87]:
ng300_info=ng300_info.query('关卡!="null"')

In [89]:
res3=ng300_info.pivot_table(values="金额USD",# 计算字段
                      index="id",# 分组的行
                      columns="关卡",# 分组的列
                      aggfunc="sum", # 聚合方式
                      fill_value=0, # 对缺失值的填充，默认为nan
                      margins=False, # 是否启用总计
                      margins_name='总计' # 总计的名称
                     )


In [84]:
# 统计关卡付费金额情况

In [117]:
res3.to_excel('大R关卡付费成长情况.xlsx',index=True)

In [123]:
def jine(x):
    qujian = ''
    if x<10:
        qujian = '0-9.9'
    elif x<20:
        qujian = '10-19.9'
    elif x<50:
        qujian = '20-49.9'
    elif x <100:
        qujian = '50-99.9'
    else:
        qujian = '大于100'
    return qujian
    
        

In [124]:
ng300_info['金额区间']=ng300_info['金额USD'].apply(lambda x : jine(x))

C:\Users\admin\AppData\Local\Temp\ipykernel_20792\9967238.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ng300_info['金额区间']=ng300_info['金额USD'].apply(lambda x : jine(x))


In [127]:
ng300_info.pivot_table(values="id",# 计算字段
                      index="关卡",# 分组的行
                      columns="金额区间",# 分组的列
                      aggfunc="nunique", # 聚合方式
                      fill_value=0, # 对缺失值的填充，默认为nan
                      margins=False, # 是否启用总计
                      margins_name='总计' # 总计的名称
                     ).to_excel('大R关卡付费成长情况.xlsx',index=True)


### 成长性

In [93]:
# 玩家拥有角色和出场情况
# 不同角色的拥有人数、拥有角色后总出场次数、本英雄出场次数


In [95]:
sql7 ="""
with tab as (
select a.*,b."#distinct_id" as "id2",if(a."name"=b."level_role",1,0) as "是否出战"  from 
(
select "#distinct_id","#event_time","name","quality"
from ta.v_event_33 
where "$part_event"  =  'role_get'
and "#country_code" = 'JP'
and "$part_date"  between '2024-01-01' and '2024-03-19') a 

left join 
--- 玩家获取角色后的出战情况
(
select "#distinct_id","#event_time","level_role"
FROM 
ta.v_event_33 
where "$part_event"  = 'Game_Level_Finish'
and "#country_code" = 'JP'
and "$part_date"  between '2024-01-01' and '2024-03-19') b 
on a."#distinct_id"=b."#distinct_id"
where a."#event_time" < b."#event_time"
)

select tab."name",tab."quality",count(distinct "#distinct_id") as "拥有人数",count("id2") as "总出战次数",sum("是否出战") as "英雄出战次数"  
from tab  
group BY tab."name",tab."quality"
"""

In [96]:
role_info = get_sqldata(token,sql7)

In [98]:
role_info.columns=['角色名称','角色等级','拥有人数','拥有角色后总出战次数','该角色出战次数']

In [110]:
role_info

,角色名称,角色等级,拥有人数,拥有角色后总出战次数,该角色出战次数
0,Eric,4,57823,3173644,129440
1,Yoyo,3,73831,3880964,99500
2,Mio,4,57766,3188237,129200
3,Vepar,5,2,126,31
4,Coola,5,13534,777813,176139
5,Alice,5,35278,2033347,577174
6,Watermelon,5,13460,789739,132537
7,Kate,3,108440,4642368,505942
8,Camila,4,58989,3279430,372854
9,Kanoka,5,13436,777431,121204


In [99]:
# 付费用户流失时，拥有的强力角色和等级

In [101]:
sql8="""
select * from (
-- 付费流失用户
select "#distinct_id","last_login_time"
from ta.v_user_33 
where date_diff('day',"last_login_time", CURRENT_TIMESTAMP) >=7
and "first_login_time" is not NULL 
and to_char("first_login_time",'yyyy-mm-dd') > '2024-01-01'
and "#distinct_id" in (
select  "#distinct_id" from ta.v_user_33 where "total_pay_amount" > 0
)
) tab1
left JOIN 
(
-- -- 流失前获得强力英雄（钻石抽+付费抽）和升级情况
select * from (
select  "#distinct_id" as "id2","name"
from ta.v_event_33 
where "$part_event"='role_get'
and "$part_date" between '2024-01-01' and '2024-03-19'
and "name" in (
'Kanade',
'Kasha',
'Lena',
'Lian',
'Luna',
'Momo',
'Snake',
'Vepar'
)
) a 
left join 

-- 升级
(
select distinct "#distinct_id" as "id3",
case when "role_id" ='503012' then 'Snake'
when "role_id" ='503013' then 'Kasha'
when "role_id" ='503014' then 'Lena'
when "role_id" ='503015' then 'Kanade'
when "role_id" ='504001' then 'Lian'
when "role_id" ='504002' then 'Momo'
when "role_id" ='504003' then 'Luna'
when "role_id" ='504004' then 'Vepar'
end as "name1"
from ta.v_event_33 
where "$part_event" = 'role_operation'
and "$part_date" between '2024-01-01' and '2024-03-19'
and "reason" = 'RoleUpgrade'
and "role_id" in (
'503012',
'503013',
'503014',
'503015',
'504001',
'504002',
'504003',
'504004'
)
) b  
on a."id2" = b."id3" and a."name" = b."name1"
)tab2
on tab1."#distinct_id" = tab2."id2"
"""

In [102]:
liushi_role_info=get_sqldata(token,sql8)

In [104]:
liushi_role_info.columns=['id','最后登录时间','id2','获得角色','id3','升级角色']

In [112]:
liushi_role_info['总流失用户']=liushi_role_info['id'].nunique()

In [113]:
liushi_role_info

,id,最后登录时间,id2,获得角色,id3,升级角色,总流失用户
0,67d77f9a6fd63143,2024-03-07 05:57:42.000,null,null,null,null,29776
1,D88EA270-B928-46AE-8D03-F9C6903872A9,2024-01-07 05:17:39.000,null,null,null,null,29776
2,66B9C929-4DF7-4263-825C-87A3C995F778,2024-01-02 12:24:39.000,null,null,null,null,29776
3,0CD732F7-5348-4245-9C92-04D0DF0C31C4,2024-02-21 12:44:25.000,null,null,null,null,29776
4,2987c2b4c0dd27bd,2024-01-26 01:29:47.000,null,null,null,null,29776
...,...,...,...,...,...,...,...
30479,dc6107c98d846a2b,2024-03-12 05:15:26.000,null,null,null,null,29776
30480,6fc0eddb2693c2a9,2024-03-12 19:06:29.000,null,null,null,null,29776
30481,25a0d083f4f5a474,2024-03-12 13:11:46.000,null,null,null,null,29776
30482,e36e51debab4e3a6,2024-03-13 00:55:23.000,null,null,null,null,29776


In [114]:
liushi_role_info.groupby(by='获得角色',as_index=False)['id2'].nunique()

,获得角色,id2
0,Kasha,506
1,Lena,1530
2,Lian,64
3,Luna,131
4,Momo,246
5,Snake,1067
6,null,1


In [115]:
liushi_role_info.groupby(by='升级角色',as_index=False)['id3'].nunique()

,升级角色,id3
0,Kasha,415
1,Lena,1176
2,Lian,56
3,Luna,126
4,Momo,230
5,Snake,651
6,null,1


In [28]:
# 首次获得s级角色，以及后续培养情况


In [147]:
role_get_sql = """
with tab as(
select "#distinct_id","name","#event_time",
rank()over(
PARTITION  by "#distinct_id"
order by "#event_time" 
) as "rank1"
from ta.v_event_33 
where "$part_event"  =  'role_get'
and "#country_code" = 'JP'
and "$part_date"  between '2023-12-01' and '2024-03-20'
and "name" in (
'Kanade',
'Kasha',
'Lena',
'Lian',
'Luna',
'Momo',
'Snake',
'Vepar'
))
select * from tab where tab."rank1" = 1
"""

In [148]:
role_upgrade_sql="""
select "#distinct_id","role_id",max("role_lv") as "角色最大等级"
from  ta.v_event_33 
where "$part_event" = 'role_operation' 
and "$part_date" between '2023-12-01' and '2024-03-20'
and "#country_code" = 'JP'
and "reason" = 'RoleUpgrade'
group by "#distinct_id","role_id"
"""

In [149]:
role_get = get_sqldata(token,role_get_sql)

In [150]:
role_get.columns  = ['user','role','time','col1']

In [160]:
role_upgrade = get_sqldata(token,role_upgrade_sql)

In [161]:
role_upgrade.columns = ['user1','role_id','role_lv']

In [162]:
role_upgrade.dropna(how='any',axis=0,inplace=True)

In [163]:
# 角色对照表
role_info = pd.read_excel('D:\Desktop\GM工具使用说明.xlsx',sheet_name='角色id')

In [164]:
role_info['id'] =role_info['id'].astype('string')
role_upgrade['role_lv'] =role_upgrade['role_lv'].astype('int')

In [165]:
role_upgrade=role_upgrade.merge(role_info,how = 'left',left_on = 'role_id',right_on = 'id')

In [166]:
role_upgrade.head()

,user1,role_id,role_lv,名称,id,技能
0,2e0914e5e7ffc1ef,501005,9,Kira,501005,自动哨兵
1,b88b9120f296996c,502007,9,Burn,502007,喷火器
2,9a24c88bd5347575,502005,9,Verena,502005,陨星
3,1846cfdeb4f4b472,501003,9,Miyako,501003,裂空刃
4,F57E4576-84D2-4744-8120-30A46C10344F,501001,9,Kaite,501001,电磁助手


In [129]:
role_get.head()

,user,role,time,col1
0,9c6dcb47dc02a632,Lena,2024-01-24 06:20:12.682,1
1,96a3bf8cdee92458,Lena,2024-01-24 16:09:32.627,1
2,81F64DCC-9045-42B9-9CBD-B08377FC5611,Kasha,2024-01-14 09:12:35.902,1
3,CBF27DF0-BCF5-40C2-B795-A05C74E3DFC1,Kasha,2024-01-14 13:14:28.542,1
4,A004F2D1-4E33-426D-8268-AF6081584588,Lena,2024-01-24 11:27:19.964,1


In [172]:
role_df = role_get.merge(role_upgrade,how='left',left_on = ['user','role'] ,right_on =['user1','名称'])

In [173]:
role_df

,user,role,time,col1,user1,role_id,role_lv,名称,id,技能
0,21679619c53dd04b,Lena,2024-02-07 09:28:26.844,1,21679619c53dd04b,503014,4.0,Lena,503014,拳魂
1,0ef99c0d72827f03,Snake,2024-02-28 17:54:02.655,1,0ef99c0d72827f03,503012,8.0,Snake,503012,磁爆蟒
2,CFE39122-E36B-47E3-8460-594EF0996C02,Snake,2024-02-28 20:50:39.495,1,CFE39122-E36B-47E3-8460-594EF0996C02,503012,9.0,Snake,503012,磁爆蟒
3,7231a08686766666,Snake,2024-02-28 22:22:09.772,1,7231a08686766666,503012,9.0,Snake,503012,磁爆蟒
4,CC026B1D-3BE7-4384-B534-CFDA3B5327A9,Lena,2024-01-25 16:19:04.320,1,CC026B1D-3BE7-4384-B534-CFDA3B5327A9,503014,6.0,Lena,503014,拳魂
...,...,...,...,...,...,...,...,...,...,...
12231,6BCA2D81-DCA6-4B16-961B-1741401D4682,Snake,2024-03-05 10:34:24.198,1,NaN,NaN,NaN,NaN,<NA>,NaN
12232,B9CF4B7F-D1B1-4BBB-ABAB-F68002924688,Snake,2024-03-05 21:53:55.576,1,B9CF4B7F-D1B1-4BBB-ABAB-F68002924688,503012,4.0,Snake,503012,磁爆蟒
12233,BC2BABB8-4E3D-431C-979B-437D8829E58A,Kasha,2024-01-02 06:36:08.968,1,BC2BABB8-4E3D-431C-979B-437D8829E58A,503013,9.0,Kasha,503013,幽夜降临
12234,3DCCEB52-705D-49EA-B651-F089AE4B3066,Kasha,2024-01-02 15:04:53.683,1,3DCCEB52-705D-49EA-B651-F089AE4B3066,503013,6.0,Kasha,503013,幽夜降临


In [174]:
role_df_gp=role_df.groupby(by='role',as_index=False)[['user','user1','role_lv']].agg({
    'user':'nunique',
    'user1':'nunique',
    'role_lv':'sum'
})

In [175]:
role_df_gp

,role,user,user1,role_lv
0,Kanade,1361,1119,7444.0
1,Kasha,3051,2555,17819.0
2,Lena,3792,3181,22276.0
3,Lian,121,107,773.0
4,Luna,174,168,1363.0
5,Momo,288,279,2132.0
6,Snake,3407,2625,17263.0
7,Vepar,41,39,252.0
